# Clustered FedAvg with two-level unweighted aggregation

Each round trains the selected clients locally, clusters their parameter updates with KMeans, averages clients equally inside each cluster, then averages cluster models equally on the server. The elbow search tests `K=2..K_MAX`.

In [ ]:
import csv
import json
import os
import random
import tarfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, models

try:
    get_ipython().run_line_magic('load_ext', 'cuml.accel')
    print('cuML acceleration enabled for sklearn KMeans.')
except Exception as error:
    print(f'cuML acceleration unavailable; using CPU sklearn KMeans: {error}')
from sklearn.cluster import KMeans

assert torch.cuda.is_available(), 'Enable the T4 GPU in Colab.'
DEVICE = torch.device('cuda')
torch.backends.cudnn.benchmark = True
print(torch.cuda.get_device_name(0))

In [ ]:
# Main configuration
DIRICHLET_ALPHA = 0.1
MIN_SAMPLES = 100

MODEL_NAME = 'cnn'  # 'cnn', 'logreg_raw', or 'resnet_linear'
DATASET_NAME = 'mnist'  # 'mnist' or 'cifar10'
#DATASET_NAME = 'cifar10'  # 'mnist' or 'cifar10'
#MODEL_NAME = 'logreg_raw'  # 'cnn', 'logreg_raw', or 'resnet_linear'
#MODEL_NAME = 'resnet_linear'  # 'cnn', 'logreg_raw', or 'resnet_linear'

NUM_CLIENTS = 25
CLIENTS_PER_ROUND = 25
ROUNDS = 100
LOCAL_EPOCHS = 5
BATCH_SIZE = 64
LEARNING_RATE = 0.01
RESNET_SIZE = 112  # use 224 for the standard ImageNet input size
EMBED_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 4096

K_MAX = 15
RUN_ID = 1      # new clustered experiment; increase for another run

PARTITION_SEED = 42
MODEL_SEED = 42
BATCH_SEED = 42
SELECTION_SEED = 43
CLUSTER_SEED = 44

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/FLasIRL(new)')
except ImportError:
    PROJECT_ROOT = Path.cwd() / 'FLasIRL(new)' if not Path('/content').exists() else Path('/content/FLasIRL(new)')

DATA_DIR = PROJECT_ROOT / 'data' / DATASET_NAME
PARTITION_DIR = DATA_DIR / 'partitions' / 'Dirichlet'
RUN_DIR = PROJECT_ROOT / 'clustered' / f'run{RUN_ID}'
for folder in [DATA_DIR, PARTITION_DIR, RUN_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RUN_DIR / 'metrics.csv'
CHECKPOINT_PATH = RUN_DIR / 'checkpoint_last.pt'
print('Run directory:', RUN_DIR)

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed % (2**32 - 1))
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

if DATASET_NAME == 'mnist':
    train_ds = datasets.MNIST(DATA_DIR, train=True, download=True)
    test_ds = datasets.MNIST(DATA_DIR, train=False, download=True)
    train_images = train_ds.data.float().div_(255.0)
    test_images = test_ds.data.float().div_(255.0)
    train_labels = train_ds.targets.long()
    test_labels = test_ds.targets.long()
    DATA_MEAN, DATA_STD = 0.1307, 0.3081
    INPUT_CHANNELS, INPUT_SIZE = 1, 28
else:
    extracted_dir = DATA_DIR / 'cifar-10-batches-py'
    if not extracted_dir.exists():
        archives = sorted(DATA_DIR.glob('*.gz'))
        if archives:
            with tarfile.open(archives[0], 'r:gz') as archive:
                archive.extractall(DATA_DIR)
    train_ds = datasets.CIFAR10(DATA_DIR, train=True, download=True)
    test_ds = datasets.CIFAR10(DATA_DIR, train=False, download=True)
    train_images = torch.from_numpy(train_ds.data).permute(0, 3, 1, 2).float().div_(255.0)
    test_images = torch.from_numpy(test_ds.data).permute(0, 3, 1, 2).float().div_(255.0)
    train_labels = torch.tensor(train_ds.targets, dtype=torch.long)
    test_labels = torch.tensor(test_ds.targets, dtype=torch.long)
    DATA_MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
    DATA_STD = torch.tensor([0.2023, 0.1994, 0.2010]).view(1, 3, 1, 1)
    INPUT_CHANNELS, INPUT_SIZE = 3, 32

if MODEL_NAME == 'cnn':
    train_features = ((train_images - DATA_MEAN) / DATA_STD)
    test_features = ((test_images - DATA_MEAN) / DATA_STD)
    if DATASET_NAME == 'mnist':
        train_features = train_features.unsqueeze(1)
        test_features = test_features.unsqueeze(1)
elif MODEL_NAME == 'logreg_raw':
    train_features = ((train_images - DATA_MEAN) / DATA_STD).reshape(len(train_images), -1)
    test_features = ((test_images - DATA_MEAN) / DATA_STD).reshape(len(test_images), -1)
else:
    train_features = None
    test_features = None
train_features = train_features.to(DEVICE)
test_features = test_features.to(DEVICE)
train_labels_gpu = train_labels.to(DEVICE)
test_labels_gpu = test_labels.to(DEVICE)

def generate_dirichlet_partition(labels, num_clients, alpha, seed, min_samples):
    labels_np = labels.cpu().numpy()
    for attempt in range(1000):
        rng = np.random.default_rng(seed + attempt)
        client_lists = [[] for _ in range(num_clients)]
        for class_id in range(10):
            class_indices = np.flatnonzero(labels_np == class_id)
            rng.shuffle(class_indices)
            proportions = rng.dirichlet(np.repeat(alpha, num_clients))
            counts = rng.multinomial(len(class_indices), proportions)
            start = 0
            for client_id, count in enumerate(counts):
                client_lists[client_id].extend(class_indices[start:start + count].tolist())
                start += count
        sizes = [len(values) for values in client_lists]
        if min(sizes) >= min_samples:
            result = []
            for values in client_lists:
                values = np.asarray(values, dtype=np.int64)
                rng.shuffle(values)
                result.append(torch.from_numpy(values.copy()).long())
            return result
    raise RuntimeError('Could not create a valid Dirichlet partition.')

partition_path = PARTITION_DIR / f'{DATASET_NAME}_dirichlet_alpha{DIRICHLET_ALPHA}_{NUM_CLIENTS}_seed{PARTITION_SEED}.pt'
if partition_path.exists():
    client_indices = torch.load(partition_path, map_location='cpu')['client_indices']
else:
    client_indices = generate_dirichlet_partition(train_labels, NUM_CLIENTS, DIRICHLET_ALPHA, PARTITION_SEED, MIN_SAMPLES)
    torch.save({'client_indices': client_indices, 'seed': PARTITION_SEED, 'alpha': DIRICHLET_ALPHA, 'min_samples': MIN_SAMPLES}, partition_path)

assert sum(len(x) for x in client_indices) == len(train_labels)
seed_everything(MODEL_SEED)

In [ ]:
# Plot the non-IID Dirichlet class distribution
client_class_counts = torch.zeros(NUM_CLIENTS, 10, dtype=torch.long)
for client_id, indices in enumerate(client_indices):
    client_class_counts[client_id] = torch.bincount(train_labels[indices], minlength=10)
client_sizes = client_class_counts.sum(dim=1).numpy()
print(f'Client sizes: min={client_sizes.min()}, max={client_sizes.max()}, mean={client_sizes.mean():.1f}, std={client_sizes.std():.1f}')

plt.figure(figsize=(14, 6))
bottom = np.zeros(NUM_CLIENTS)
for class_id in range(10):
    values = client_class_counts[:, class_id].numpy()
    plt.bar(np.arange(NUM_CLIENTS), values, bottom=bottom, label=f'Class {class_id}')
    bottom += values
plt.xlabel('Client ID')
plt.ylabel('Number of samples')
plt.title(f'Non-IID Dirichlet distribution (alpha={DIRICHLET_ALPHA})')
plt.xticks(np.arange(NUM_CLIENTS))
plt.legend(ncol=10)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(INPUT_CHANNELS, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        side = INPUT_SIZE // 4
        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(64 * side * side, 128), nn.ReLU(), nn.Linear(128, 10))

    def forward(self, x):
        return self.classifier(self.features(x))


class RawLogisticRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(INPUT_CHANNELS * INPUT_SIZE * INPUT_SIZE, 10)

    def forward(self, x):
        return self.linear(x)


class LinearProbe(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(512, 10)

    def forward(self, x):
        return self.linear(x)


def preprocess_resnet(images):
    x = images.unsqueeze(1).repeat(1, 3, 1, 1) if DATASET_NAME == 'mnist' else images
    x = F.interpolate(x, size=(RESNET_SIZE, RESNET_SIZE), mode='bilinear', align_corners=False)
    mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
    return (x - mean) / std


if MODEL_NAME == 'resnet_linear':
    cache_key = f'{DATASET_NAME}_resnet18_resize{RESNET_SIZE}_imagenet_norm'
    train_cache = DATA_DIR / f'{cache_key}_train.pt'
    test_cache = DATA_DIR / f'{cache_key}_test.pt'
    if train_cache.exists() and test_cache.exists():
        train_features = torch.load(train_cache, map_location='cpu')
        test_features = torch.load(test_cache, map_location='cpu')
    else:
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).to(DEVICE)
        backbone.fc = nn.Identity()
        backbone.eval()
        with torch.no_grad():
            train_features = torch.cat([backbone(preprocess_resnet(train_images[i:i + EMBED_BATCH_SIZE].to(DEVICE))).cpu() for i in range(0, len(train_images), EMBED_BATCH_SIZE)])
            test_features = torch.cat([backbone(preprocess_resnet(test_images[i:i + EMBED_BATCH_SIZE].to(DEVICE))).cpu() for i in range(0, len(test_images), EMBED_BATCH_SIZE)])
        torch.save(train_features, train_cache)
        torch.save(test_features, test_cache)
        del backbone
        torch.cuda.empty_cache()
    train_features = train_features.to(DEVICE)
    test_features = test_features.to(DEVICE)


def build_model():
    if MODEL_NAME == 'cnn':
        model = SmallCNN()
    elif MODEL_NAME == 'logreg_raw':
        model = RawLogisticRegression()
    else:
        model = LinearProbe()
    return model.to(DEVICE)


def select_clients(round_id):
    if CLIENTS_PER_ROUND == NUM_CLIENTS:
        return list(range(NUM_CLIENTS))
    generator = torch.Generator(device='cpu').manual_seed(SELECTION_SEED + round_id)
    return torch.randperm(NUM_CLIENTS, generator=generator)[:CLIENTS_PER_ROUND].tolist()


def batch_seed(round_id, client_id, epoch):
    return BATCH_SEED + round_id * 1_000_000 + client_id * 1_000 + epoch


def local_train(model, global_state, client_id, round_id):
    model.load_state_dict(global_state)
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE)
    indices = client_indices[client_id]
    for epoch in range(LOCAL_EPOCHS):
        generator = torch.Generator(device='cpu').manual_seed(batch_seed(round_id, client_id, epoch))
        permutation = indices[torch.randperm(len(indices), generator=generator)].to(DEVICE)
        for start in range(0, len(permutation), BATCH_SIZE):
            batch = permutation[start:start + BATCH_SIZE]
            loss = F.cross_entropy(model(train_features[batch]), train_labels_gpu[batch])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
    return {name: p.detach().clone() for name, p in model.named_parameters()}, len(indices)


def flatten_state(state, names):
    return torch.cat([state[name].reshape(-1) for name in names])


def state_from_vector(vector, template, names):
    result, offset = {}, 0
    for name in names:
        size = template[name].numel()
        result[name] = vector[offset:offset + size].reshape(template[name].shape)
        offset += size
    return result


def choose_elbow_k(vectors, max_k, round_id):
    max_k = min(max_k, len(vectors))
    if max_k < 2:
        return 1, np.array([1]), np.array([0.0])
    ks = np.arange(2, max_k + 1)
    inertias = []
    for k in ks:
        # Keep 10 initializations for stable elbow selection. cuML accelerates this on GPU in Colab.
        km = KMeans(n_clusters=int(k), n_init=10, random_state=CLUSTER_SEED + round_id)
        km.fit(vectors)
        inertias.append(km.inertia_)
    points = np.column_stack([ks.astype(float), np.asarray(inertias)])
    normalized = (points - points.min(axis=0)) / np.maximum(points.max(axis=0) - points.min(axis=0), 1e-12)
    line = normalized[-1] - normalized[0]
    distances = np.abs(line[0] * (normalized[:, 1] - normalized[0, 1]) - line[1] * (normalized[:, 0] - normalized[0, 0])) / np.linalg.norm(line)
    return int(ks[int(np.argmax(distances))]), ks, np.asarray(inertias)


def clustered_aggregate(client_states, global_state, names, round_id):
    global_vector = flatten_state(global_state, names)
    update_vectors = torch.stack([flatten_state(state, names) - global_vector for state in client_states]).cpu().numpy()
    selected_k, ks, inertias = choose_elbow_k(update_vectors, K_MAX, round_id)
    clustering = KMeans(n_clusters=selected_k, n_init=10, random_state=CLUSTER_SEED + round_id).fit(update_vectors)
    cluster_models = []
    for cluster_id in range(selected_k):
        members = np.flatnonzero(clustering.labels_ == cluster_id)
        # Equal weight for every client inside this cluster.
        cluster_vector = torch.stack([flatten_state(client_states[i], names) for i in members]).mean(dim=0)
        cluster_models.append(state_from_vector(cluster_vector, global_state, names))
    # Equal weight for every cluster on the server.
    server_vector = torch.stack([flatten_state(state, names) for state in cluster_models]).mean(dim=0)
    return state_from_vector(server_vector, global_state, names), selected_k, ks, inertias, clustering.labels_


@torch.no_grad()
def evaluate(model, features, labels):
    model.eval(); loss_sum = 0.0; correct = 0
    for start in range(0, len(labels), EVAL_BATCH_SIZE):
        logits = model(features[start:start + EVAL_BATCH_SIZE]); y = labels[start:start + EVAL_BATCH_SIZE]
        loss_sum += F.cross_entropy(logits, y, reduction='sum').item()
        correct += (logits.argmax(1) == y).sum().item()
    return loss_sum / len(labels), correct / len(labels)

global_model = build_model()
worker_model = build_model()
names = [name for name, _ in global_model.named_parameters()]
print(global_model)

In [ ]:
METRIC_FIELDS = ['round', 'global_train_loss', 'test_loss', 'test_accuracy', 'num_selected_clients', 'selected_k', 'selected_clients']

def append_metric(row):
    exists = METRICS_PATH.exists()
    with METRICS_PATH.open('a', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=METRIC_FIELDS)
        if not exists: writer.writeheader()
        writer.writerow(row); handle.flush(); os.fsync(handle.fileno())

for round_id in range(1, ROUNDS + 1):
    start_time = time.time()
    selected_clients = select_clients(round_id)
    global_state = {name: p.detach().clone() for name, p in global_model.named_parameters()}
    client_states = []
    for client_id in selected_clients:
        state, _ = local_train(worker_model, global_state, client_id, round_id)
        client_states.append(state)
    new_state, selected_k, ks, inertias, labels = clustered_aggregate(client_states, global_state, names, round_id)
    global_model.load_state_dict(new_state, strict=True)
    train_loss, _ = evaluate(global_model, train_features, train_labels_gpu)
    test_loss, test_accuracy = evaluate(global_model, test_features, test_labels_gpu)
    append_metric({'round': round_id, 'global_train_loss': train_loss, 'test_loss': test_loss, 'test_accuracy': test_accuracy, 'num_selected_clients': len(selected_clients), 'selected_k': selected_k, 'selected_clients': json.dumps(selected_clients)})
    torch.save({'round': round_id, 'model_state': {n: p.detach().cpu() for n, p in global_model.named_parameters()}, 'selected_k': selected_k, 'selected_clients': selected_clients}, CHECKPOINT_PATH)
    print(f'Round {round_id:02d}/{ROUNDS} | selected K={selected_k} | test accuracy={test_accuracy:.2%} | {time.time() - start_time:.1f}s')

torch.save({n: p.detach().cpu() for n, p in global_model.named_parameters()}, RUN_DIR / 'final_model_parameters.pt')

In [ ]:
# Plot clustered FedAvg metrics
if METRICS_PATH.exists():
    with METRICS_PATH.open(newline='') as handle:
        rows = list(csv.DictReader(handle))
    rounds = [int(row['round']) for row in rows]
    accuracy = [100 * float(row['test_accuracy']) for row in rows]
    selected_k = [int(row['selected_k']) for row in rows]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(rounds, accuracy, marker='o', markersize=3)
    axes[0].set(xlabel='FedAvg round', ylabel='Test accuracy (%)', title='Clustered FedAvg accuracy')
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(rounds, selected_k, marker='o', markersize=3)
    axes[1].set(xlabel='FedAvg round', ylabel='Selected K', title='Elbow-selected number of clusters')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print(f'No metrics found at {METRICS_PATH}')